# Multiple testing - the blessing of dimensionality

## Background

In the [causal model selection notebook](1_causal_model_selection.ipynb), we saw that testing for a causal interaction $X\to Y$ (using an instrumental variable $Z$) requires combining evidence from mulitple hypothesis tests. In that notebook, we basically eye-balled the p-values and gave thumbs-up or thumbs-down for a causal relation. Not only is this practically infeasible when testing causality between ten-thousands of pairs in omics data, we would also like to quantify the statistical significance and uncertainty of each finding. This turns out to be a non-trivial question, because:

- Both the sufficient and necessary condition are *combinations* of statistical tests.
- The necessary condition includes a test where *accepting* the null hypothesis is a *positive* finding, but a non-significant p-value (no evidence for rejecting the null hypothesis) does not imply significant evidence against the alternative hypothesis.

The aim of this notebook is to explain how we go from classical hypothesis tests to expressing the probability of a model being true, by exploiting that we are testing thousands of genes at once, that is by exploiting the blessing of dimensionality!

[BioFindr](https://github.com/tmichoel/BioFindr.jl) is a software for doing exactly this. It allows to do causal model selection by combining the following hypothesis tests:

![BioFindr tests](fig-findr-tests-new.png)

## Setup the environment

In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
import matplotlib.pyplot as plt
import os

# Setup PyJulia for calling the Julia BioFindr package.
# Julia must be installed with BioFindr and its dependencies.
# Install PyJulia with: pip install julia
# Then in Python: import julia; julia.install()
from julia.api import Julia
jl = Julia(compiled_modules=False)
from julia import Main

print("PyJulia setup complete")

In [ ]:
# Load only the Julia packages needed for BioFindr calls.
# DataFrames is needed to pass data to BioFindr; Distributions for pdf() on returned objects.
Main.eval("""
using BioFindr
using DataFrames
using Distributions
using Graphs
""")
print("Julia packages loaded")

## Load data

This tutorial uses [preprocessed data files](https://github.com/lingfeiwang/findr-data-geuvadis) from the [GEUVADIS study](https://doi.org/10.1038/nature12531). In particular, we use genotype and mRNA expression data from the same set of samples:

In [ ]:
# Determine path to the data directory.
# The data should be stored at: <project_root>/data/processed/findr-data-geuvadis/
notebook_dir = os.path.abspath('.')
if os.path.basename(notebook_dir) == 'notebooks':
    project_root = os.path.dirname(notebook_dir)
else:
    project_root = notebook_dir
data_dir = os.path.join(project_root, 'data', 'processed', 'findr-data-geuvadis')

def read_arrow(path):
    """Read an Arrow IPC file into a pandas DataFrame."""
    try:
        return pa.ipc.open_file(path).read_all().to_pandas()
    except pa.lib.ArrowInvalid:
        return pa.ipc.open_stream(path).read_all().to_pandas()

dgt = read_arrow(os.path.join(data_dir, 'dgt.arrow'))
dt  = read_arrow(os.path.join(data_dir, 'dt.arrow'))

print(f"Genotype data (dgt): {dgt.shape[0]} samples x {dgt.shape[1]} SNPs")
print(f"Expression data (dt): {dt.shape[0]} samples x {dt.shape[1]} genes")

We apply an [inverse normal transformation](https://lab.michoel.info/BioFindr.jl/stable/inference/#BioFindr.supernormalize) such that the expression profiles of all genes are normally distributed:

In [ ]:
# Convert the expression DataFrame to a Julia DataFrame and call BioFindr.supernormalize.
# The result Yt is a Julia Matrix kept in Julia for subsequent BioFindr calls.
Main._dt_values  = dt.values.copy()
Main._dt_columns = dt.columns.tolist()
Main.eval("""
Yt = BioFindr.supernormalize(DataFrame(_dt_values, _dt_columns))
""")
print(f"Normalized expression matrix size: {Main.eval('size(Yt)')} (samples x genes)")

The [preprocessed GEUVADIS data](https://github.com/lingfeiwang/findr-data-geuvadis) has been organized such that each column of the genotype data is the strongest eQTLs for the corresponding column in the matching expression data. Usually however, eQTL mapping data will be available in the form of a table with variant IDs, gene IDs, and various eQTL association statistics. Let's artificially create such a table for our data:

In [ ]:
dpt = pd.DataFrame({
    'SNP_ID': dgt.columns.tolist(),
    'GENE_ID': dt.columns[:dgt.shape[1]].tolist()
})
dpt.head()

In this analysis we will use one SNP as an instrumental variable $Z$ and test the association $Z\to Y$ for all genes $Y$. Here's a good example SNP to use:

In [ ]:
snp_name = "rs6504337"
Z = dgt[snp_name].values  # numpy array of genotype values
print(f"SNP: {snp_name}")
print(f"Number of samples: {len(Z)}")
print(f"Unique genotype values: {np.unique(Z)}")

For later use, we store the gene for which this SNP is an eQTL. Consistent with the [causal model selection notebook](1_causal_model_selection.ipynb), we call this gene's expression variable $X$:

In [ ]:
gene_name = dpt.loc[dpt['SNP_ID'] == snp_name, 'GENE_ID'].iloc[0]
gene_idx  = dt.columns.get_loc(gene_name) + 1  # Julia uses 1-based indexing
print(f"Gene associated with {snp_name}: {gene_name} (column index {gene_idx} in Julia)")

# Extract the corresponding column from the already-computed Yt matrix in Julia
Main._gene_idx = gene_idx
Main.eval("X = Yt[:, _gene_idx]")

Lastly, we need the number of samples in our data, the number of unique genotypes in $Z$, and a randomly shuffled version of $Z$:

In [ ]:
ns = len(Z)
ng = len(np.unique(Z))
print(f"Number of samples: ns = {ns}")
print(f"Number of unique genotypes: ng = {ng}")

In [ ]:
rng = np.random.default_rng(123)
Z_rand = rng.permutation(Z)

## P-value refresher

Recall that in statistical hypothesis testing, we have a null hypothesis $H_0$ and an alternative hypothesis $H_1$. For instance, if we test the association between $Z$ and $Y$, the null hypothesis is that there is no association and the alternative that there is an association. To quantify evidence for or against the null hypothesis, we need a *test statistic*. In the [BioFindr](https://github.com/tmichoel/BioFindr.jl) software all hypothesis tests are [likelihood ratio tests](https://en.wikipedia.org/wiki/Likelihood-ratio_test), and the test statistics are *log-likelihood ratios (LLRs).* The null distribution of these LLRs are [known analytically](https://lab.michoel.info/BioFindr.jl/stable/randomLLR/).

The [p-value](https://en.wikipedia.org/wiki/P-value) is defined as the probability of obtaining test results at least as extreme as the result actually observed under the assumption that the null hypothesis is correct. In other words, if we observe $LLR=x$, then

$$
\text{p-value}(x) = P(LLR\geq x \mid H_0)
$$

Let's compute the LLRs and their p-values under the null hypothesis for the association or *"linkage"* test $Z\to Y$:

In [ ]:
# Pass inputs from Python to Julia, then call BioFindr
Main._Z  = Z
Main._ns = ns
Main._ng = ng
Main.eval("""
llr   = BioFindr.realLLR_col(Yt, _Z)
pnull = BioFindr.nullpval(llr, _ns, _ng, :link)
""")

llr   = np.array(Main.llr)
pnull = np.array(Main.pnull)
print(f"Computed {len(llr)} LLR values (one per gene)")
print(f"LLR range: [{llr.min():.3f}, {llr.max():.3f}]")

We can do the same for the randomly shuffled $Z$:

In [ ]:
Main._Z_rand = Z_rand
Main.eval("""
llr_rand   = BioFindr.realLLR_col(Yt, _Z_rand)
pnull_rand = BioFindr.nullpval(llr_rand, _ns, _ng, :link)
""")

llr_rand   = np.array(Main.llr_rand)
pnull_rand = np.array(Main.pnull_rand)

An important property of p-values is that under the null hypothesis, p-values are uniformly distributed. We can verify this by plotting the histogram of p-values for the randomly shuffled instruments (left histogram). By contrast the observed distribution in the real data shows an inflation of small p-values.

<div style="background-color:LightYellow; color:black">
<h3>Exercise</h3> 
     What does the inflation of small p-values mean? Read <a href="http://varianceexplained.org/statistics/interpreting-pvalue-histogram/">how to interpret p-value histograms</a> for clues!
</div>

In [ ]:
Main._pnull = pnull
Main.eval("pi0_val = BioFindr.pi0est(_pnull)")
pi0 = float(Main.pi0_val)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: null distribution (randomly shuffled Z)
axes[0].hist(pnull_rand, bins=10, density=True, color='steelblue', edgecolor='white')
axes[0].set_xlabel('p-value')
axes[0].set_ylabel('density')
axes[0].set_title('Null distribution (shuffled Z)')
axes[0].set_xlim(0, 1)

# Right: observed distribution with pi0 line
axes[1].hist(pnull, bins=10, density=True, color='steelblue', edgecolor='white')
axes[1].axhline(y=pi0, color='red', linewidth=2, label=f'pi0 = {pi0:.3f}')
axes[1].set_xlabel('p-value')
axes[1].set_ylabel('density')
axes[1].set_title('Observed distribution')
axes[1].legend()
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()

## Bayes theorem and model probabilities

A p-value gives a probability of observing a test statistic assuming a certain model (the null model) is true. In model selection, we are interested in the opposite: the probability that a model (or hypothesis) is true given the observed test statistic. The relation between those two is given by Bayes's theorem:

$$
P(H_0 = \text{true} \mid LLR= x ) = \frac{p(LLR= x\mid H_0)P(H_0 = \text{true})}{p(LLR= x)} = 1 - P(H_1 = \text{true} \mid LLR= x )
$$

In this expression:

- $P(H_0 = \text{true})$ is the prior probability of observing the null model
- $p(LLR=x\mid H_0)$ is the null distribution of the LLR test statistic
- $p(LLR= x)$ is the observed distribution of the LLR test statistic

The null distribution in our case is known [analytically](https://lab.michoel.info/BioFindr.jl/stable/randomLLR/). The observed distribution can be fitted from a histogram because we test thousands of $Y$'s for every $Z$ (the blessing of dimensionality!). Hence we only need to estimate the prior probability $\pi_0=P(H_0 = \text{true})$. This is done in the same way as in Storey & Tibshirani's [q-value](https://en.wikipedia.org/wiki/Q-value_(statistics)) method.

The equation above is equivalent to saying that the LLR distribution is a mixture of a null and alternative distribution:

$$
p(LLR= x) = \pi_0\; p(LLR= x\mid H_0) + (1-\pi_0)\; p(LLR= x\mid H_1).
$$

We can also express this property in terms of p-values:

$$
f(p) = \pi_0 f_0(p) + (1-\pi_0) f_1(p)
$$

where $f(p)$ is the probability density function (p.d.f.) of the observed $p$-value distribution, $f_0$ and $f_1$ are the p.d.f. of the $p$-value distribution under the null and alternative distribution.

1. $p$-values are uniformly distributed under the null hypothesis, that is

   $$
   f_0(p) = 1
   $$

2. $p$-values close to 1 almost surely come from the null distribution, that is,

   $$
  \begin{aligned}
    \lim_{p\to 1} f_1(p) &= 0 \\\\
    \lim_{p\to 1} f(p) &= \pi_0 \lim_{p\to 1} f_0(p) = \pi_0
  \end{aligned}
  $$

Hence in the observed $p$-value histogram, $\pi_0$ can be estimated from the height of the flat region near $p\approx 1$. $\pi_0$ was estimated above and marked with a red horizontal line:

In [ ]:
print(f"pi0 = {pi0:.4f}")

The null distribution $p(LLR= x\mid H_0)$ is obtained as follows:

In [ ]:
Main.eval("dnull = BioFindr.nulldist(_ns, _ng, :link)")
print("Null distribution:", Main.eval("dnull"))

The observed distribution $p(LLR= x)$ is fitted as follows:

In [ ]:
Main.eval("pp, dmix = BioFindr.fit_mixdist_mom(llr, _ns, _ng, :link)")
pp = float(Main.eval("pp"))
print(f"Mixture parameter pp = {pp:.4f}")

In [ ]:
# lval is computed in Python; PDFs are computed in Julia because dnull and dmix are
# Julia distribution objects returned by BioFindr. prob_H1 is computed in Python.
lval = np.linspace(0, np.max(llr), 500)
Main._lval = lval
Main.eval("""
pdf_null = pdf.(dnull, _lval)
pdf_real = pdf.(dmix, _lval)
""")
pdf_null = np.array(Main.pdf_null)
pdf_real = np.array(Main.pdf_real)

prob_H1 = 1 - pi0 * pdf_null / pdf_real

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(llr, bins=50, density=True, alpha=0.5, color='steelblue', label='Observed histogram')
ax.plot(lval, pdf_real,       color='red',   linewidth=1.5, label='Fitted p(LLR=x)')
ax.plot(lval, pi0 * pdf_null, color='black', linewidth=1.5, label='Analytic pi0 * p(LLR=x|H0)')
ax.plot(lval, 1e3 * prob_H1,  color='green', linewidth=1.5, label='P(H1=true | LLR=x) (x1000)')
ax.set_xlabel('LLR')
ax.set_ylabel('pdf')
ax.legend()
plt.tight_layout()
plt.show()

The log-likelihood ratios for tests 2 to 5 from the figure above can be computed as:

In [ ]:
# X and Yt are already in Julia from earlier cells; pass Z from Python
Main.eval("llr2, llr3, llr4, llr5 = BioFindr.realLLR_col(Yt, X, _Z)")

llr2 = np.array(Main.llr2)
llr3 = np.array(Main.llr3)
llr4 = np.array(Main.llr4)
llr5 = np.array(Main.llr5)

print(f"LLR shapes - Test 2: {llr2.shape}, Test 3: {llr3.shape}, Test 4: {llr4.shape}, Test 5: {llr5.shape}")

<div style="background-color:LightYellow; color:black">
<h3>Exercise</h3> 
     Follow the steps above to compute the probabilities of the null and alternative model for test 3 being true. The abbreviation for test 3 is <b>:med</b> (for "mediation"). Then combine null or alternative probabilities for test 2 and test 3 to compute the probability that model 1a (that is $Z\to X\to Y$ without hidden confounders) from the previous notebook is true. Find the gene $Y$ (different from $X$!) that has highest probability for this model, and verify visually (using boxplots and linear regressions) that this gene indeed meets the sufficient condition for a causal interaction $X\to Y$, that is, $Z$ is associated with $X$, $Z$ is associated with $Y$, and the association between $Z$ and $Y$ disappears after conditioning on $X$.
</div>